# 04 - Fetch Game Stats and Season Stats

**Database: MySQL** (`NHL_Analytics_Hub`). Run notebooks 01, 02 and 03 first.

**Part A - `game_stats`** (per-game skater stats): `GET /v1/gamecenter/{game_id}/boxscore`, for finished games.

**Part B - `skater_season_stats` / `goalie_season_stats`**: `GET /v1/player/{player_id}/landing`,
routed by the player's position (G = goalie table, anything else = skater table).

Every game and player is saved in its own small transaction, so you can stop and re-run at any time:
Part A skips games already loaded, and Part B overwrites with fresh numbers.

In [ ]:
%pip install requests pandas mysql-connector-python

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import mysql.connector
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

REQUEST_DELAY = 0.4

# Same settings as notebooks 01 and 02
DB_HOST     = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT     = int(os.getenv("DB_PORT", "3306"))
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "root")   # <-- change to YOUR MySQL password
DB_NAME     = os.getenv("DB_NAME", "NHL_Analytics_Hub")

def get_connection():
    return mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD,
        database=DB_NAME, connection_timeout=15,
    )

session = requests.Session()
_retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
               allowed_methods=["GET"], respect_retry_after_header=True)
session.mount("https://", HTTPAdapter(max_retries=_retry))

def get_json(url):
    resp = session.get(url, timeout=(10, 30))
    resp.raise_for_status()
    data = resp.json()
    if not isinstance(data, dict):
        raise ValueError("Unexpected API response")
    return data

BOXSCORE_URL = "https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"
PLAYER_LANDING_URL = "https://api-web.nhle.com/v1/player/{player_id}/landing"

REFRESH_ALL_GAMES = False   # True = re-download every finished game, not just new ones

In [ ]:
def text_val(v, limit=None):
    if isinstance(v, dict):
        v = v.get("default")
    if v is None:
        return None
    v = str(v).strip()
    if not v:
        return None
    return v[:limit] if limit else v

def int_val(v):
    try:
        return int(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def float_val(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def date_val(v):
    return v if isinstance(v, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", v) else None

## Connect and create the three tables

In [ ]:
conn = get_connection()
cur = conn.cursor()

TABLES = {
"game_stats": """
CREATE TABLE IF NOT EXISTS game_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    game_id BIGINT NOT NULL,
    player_id BIGINT NOT NULL,
    team_id INT NOT NULL,
    goals INT, assists INT, points INT, shots_on_goal INT, penalty_min INT,
    toi VARCHAR(10), plus_minus INT,
    UNIQUE (game_id, player_id),
    FOREIGN KEY (game_id) REFERENCES games(game_id),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
"skater_season_stats": """
CREATE TABLE IF NOT EXISTS skater_season_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    player_id BIGINT NOT NULL,
    season VARCHAR(20) NOT NULL,
    team_id INT NOT NULL,
    games_played INT, goals INT, assists INT, points INT,
    plus_minus INT, penalty_min INT, shots INT, avg_toi VARCHAR(10),
    UNIQUE (player_id, season),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
"goalie_season_stats": """
CREATE TABLE IF NOT EXISTS goalie_season_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    player_id BIGINT NOT NULL,
    season VARCHAR(20) NOT NULL,
    team_id INT NOT NULL,
    games_played INT, wins INT, losses INT, ot_losses INT,
    save_pct FLOAT, goals_against_avg FLOAT, shutouts INT, saves INT,
    UNIQUE (player_id, season),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
}
for name, sql in TABLES.items():
    cur.execute(sql)
conn.commit()
print("Tables ready:", ", ".join(TABLES))

---
# Part A - Game Stats (skaters, per game)

## Load finished games

Finished games have state `OFF` **or** `FINAL`. The original notebook only used `OFF`, which silently skips
games that just ended.

In [ ]:
cur.execute("""
    SELECT g.game_id, g.home_team_id, g.away_team_id
    FROM games g
    WHERE g.game_state IN ('OFF', 'FINAL')
      AND (%s OR NOT EXISTS (SELECT 1 FROM game_stats gs WHERE gs.game_id = g.game_id))
    ORDER BY g.game_date
""", (1 if REFRESH_ALL_GAMES else 0,))
games_to_load = [(int(a), int(b), int(c)) for a, b, c in cur.fetchall()]

cur.execute("SELECT player_id FROM players")
known_players = {int(r[0]) for r in cur.fetchall()}

print(f"{len(games_to_load)} finished games to load, {len(known_players)} known players")
if not known_players:
    raise RuntimeError("players table is empty - run notebook 02 first.")

## Inspect one boxscore before writing the parser

In [ ]:
if not games_to_load:
    print("Nothing to load (all finished games already in game_stats, or none finished yet).")
    sample_box = None
else:
    sample_box = None
    for gid, _, _ in games_to_load[:5]:
        try:
            candidate = get_json(BOXSCORE_URL.format(game_id=gid))
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"Game {gid}: request failed - {e}")
            continue
        away = (candidate.get("playerByGameStats") or {}).get("awayTeam") or {}
        if away.get("forwards"):
            sample_box, sample_gid = candidate, gid
            print(f"Using game {gid} as the sample.")
            break
        print(f"Game {gid}: no skater data - skipping.")

if sample_box:
    print("playerByGameStats keys:", list(sample_box["playerByGameStats"].keys()))
    print("awayTeam sub-keys:", list(sample_box["playerByGameStats"]["awayTeam"].keys()))
    display(sample_box["playerByGameStats"]["awayTeam"]["forwards"][0])

In [ ]:
def parse_boxscore(box_json, game_id, home_team_id, away_team_id):
    """Skaters only (forwards + defense). Returns (rows, skipped_unknown_players).

    Rows for players that are not in our players table are skipped: the table has a
    foreign key to players, and notebook 02 only loads CURRENT rosters, so players who were
    traded, retired or sent down would otherwise break the insert.
    """
    rows, skipped = [], 0
    stats = box_json.get("playerByGameStats") or {}
    for side, team_id in (("homeTeam", home_team_id), ("awayTeam", away_team_id)):
        side_stats = stats.get(side) or {}
        for group in ("forwards", "defense"):
            for p in side_stats.get(group) or []:
                pid = int_val(p.get("playerId"))
                if pid is None:
                    continue
                if pid not in known_players:
                    skipped += 1
                    continue
                goals, assists = int_val(p.get("goals")), int_val(p.get("assists"))
                points = int_val(p.get("points"))
                if points is None and goals is not None and assists is not None:
                    points = goals + assists
                rows.append((
                    int(game_id), pid, int(team_id), goals, assists, points,
                    int_val(p.get("sog", p.get("shots"))), int_val(p.get("pim")),
                    text_val(p.get("toi"), 10), int_val(p.get("plusMinus")),
                ))
    return rows, skipped

if sample_box:
    _, h, a = next(g for g in games_to_load if g[0] == sample_gid)
    test_rows, test_skipped = parse_boxscore(sample_box, sample_gid, h, a)
    print(f"Parsed {len(test_rows)} skater rows for game {sample_gid} ({test_skipped} skipped: not in players)")
    display(pd.DataFrame(test_rows, columns=["game_id", "player_id", "team_id", "goals", "assists",
                                             "points", "shots_on_goal", "penalty_min", "toi", "plus_minus"]).head())

## Loop over finished games (one small transaction per game, plus a retry pass)

In [ ]:
GAME_UPSERT = """
INSERT INTO game_stats
    (game_id, player_id, team_id, goals, assists, points, shots_on_goal, penalty_min, toi, plus_minus)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_id=VALUES(team_id), goals=VALUES(goals), assists=VALUES(assists), points=VALUES(points),
    shots_on_goal=VALUES(shots_on_goal), penalty_min=VALUES(penalty_min),
    toi=VALUES(toi), plus_minus=VALUES(plus_minus)
"""

totals = {"rows": 0, "skipped": 0}

def load_games(batch):
    failed = []
    start = time.time()
    for i, (gid, home_id, away_id) in enumerate(batch, 1):
        try:
            rows, skipped = parse_boxscore(get_json(BOXSCORE_URL.format(game_id=gid)), gid, home_id, away_id)
            cur.executemany(GAME_UPSERT, rows)
            conn.commit()
            totals["rows"] += len(rows)
            totals["skipped"] += skipped
        except Exception as e:
            conn.rollback()
            print(f"Game {gid}: FAILED - {type(e).__name__}: {e}")
            failed.append((gid, home_id, away_id))
        if i % 25 == 0:
            print(f"[{i}/{len(batch)}] {totals['rows']} rows, {time.time() - start:.0f}s elapsed")
        time.sleep(REQUEST_DELAY)
    return failed

failed_games = load_games(games_to_load)
if failed_games:
    print(f"\nRetrying {len(failed_games)} failed games...")
    time.sleep(3)
    failed_games = load_games(failed_games)

print(f"\nLoaded {totals['rows']} skater-game rows. "
      f"Skipped {totals['skipped']} rows for players not in the players table.")
if failed_games:
    print("Still failing (re-run this cell):", [g[0] for g in failed_games])

## Sanity check

In [ ]:
cur.execute("SELECT COUNT(*) FROM game_stats")
print("Rows in game_stats:", cur.fetchone()[0])

# GROUP BY must include every non-aggregated column (MySQL's default ONLY_FULL_GROUP_BY mode)
cur.execute("""
    SELECT p.first_name, p.last_name, SUM(gs.goals) AS total_goals,
           SUM(gs.points) AS total_points, COUNT(*) AS games
    FROM game_stats gs
    JOIN players p ON gs.player_id = p.player_id
    GROUP BY p.player_id, p.first_name, p.last_name
    ORDER BY total_points DESC
    LIMIT 10
""")
display(pd.DataFrame(cur.fetchall(), columns=["first_name", "last_name", "total_goals", "total_points", "games"]))

---
# Part B - Season Stats (skaters and goalies)

## Load all players

In [ ]:
cur.execute("SELECT player_id, team_id, position FROM players ORDER BY player_id")
players_list = [(int(a), int(b), c) for a, b, c in cur.fetchall()]
print(f"{len(players_list)} players loaded")

## Inspect one skater and one goalie landing response

In [ ]:
sample_skater_id = next((pid for pid, _, pos in players_list if pos != "G"), None)
sample_goalie_id = next((pid for pid, _, pos in players_list if pos == "G"), None)
if sample_skater_id is None:
    raise RuntimeError("No skaters in players table - run notebook 02 first.")

sample_skater = get_json(PLAYER_LANDING_URL.format(player_id=sample_skater_id))
print("SKATER position:", sample_skater.get("position"),
      "| featured season:", (sample_skater.get("featuredStats") or {}).get("season"))
print((sample_skater.get("featuredStats") or {}).get("regularSeason", {}).get("subSeason"))

if sample_goalie_id:
    sample_goalie = get_json(PLAYER_LANDING_URL.format(player_id=sample_goalie_id))
    print("\nGOALIE position:", sample_goalie.get("position"))
    print("subSeason keys:", (sample_goalie.get("featuredStats") or {}).get("regularSeason", {}).get("subSeason"))
    totals_sample = [t for t in sample_goalie.get("seasonTotals", [])
                     if t.get("leagueAbbrev") == "NHL" and t.get("gameTypeId") == 2]
    print("Latest NHL seasonTotals entry:", totals_sample[-1] if totals_sample else None)

## Parse functions

The goalie `saves` value is not always given directly. The parser uses `saves` if present,
otherwise `shotsAgainst - goalsAgainst`, otherwise `NULL`. Check the printed goalie output above
and adjust the field names if your API response differs.

In [ ]:
def latest_nhl_total(data, season):
    """First regular-season NHL entry in seasonTotals for this season (None if absent)."""
    for entry in data.get("seasonTotals") or []:
        if (entry.get("season") == season and entry.get("leagueAbbrev") == "NHL"
                and entry.get("gameTypeId") == 2):
            return entry
    return None

def parse_skater_stats(data, player_id, team_id):
    featured = data.get("featuredStats") or {}
    season = featured.get("season")
    sub = (featured.get("regularSeason") or {}).get("subSeason")
    if not season or not sub:
        return None
    total = latest_nhl_total(data, season) or {}
    return (
        player_id, str(season), team_id,
        int_val(sub.get("gamesPlayed")), int_val(sub.get("goals")), int_val(sub.get("assists")),
        int_val(sub.get("points")), int_val(sub.get("plusMinus")), int_val(sub.get("pim")),
        int_val(sub.get("shots")), text_val(total.get("avgToi"), 10),
    )

def parse_goalie_stats(data, player_id, team_id):
    featured = data.get("featuredStats") or {}
    season = featured.get("season")
    sub = (featured.get("regularSeason") or {}).get("subSeason")
    if not season or not sub:
        return None
    total = latest_nhl_total(data, season) or {}

    saves = int_val(sub.get("saves"))
    if saves is None:
        shots_against = int_val(sub.get("shotsAgainst", total.get("shotsAgainst")))
        goals_against = int_val(sub.get("goalsAgainst", total.get("goalsAgainst")))
        if shots_against is not None and goals_against is not None:
            saves = shots_against - goals_against

    return (
        player_id, str(season), team_id,
        int_val(sub.get("gamesPlayed")), int_val(sub.get("wins")), int_val(sub.get("losses")),
        int_val(sub.get("otLosses")), float_val(sub.get("savePctg")),
        float_val(sub.get("goalsAgainstAvg")), int_val(sub.get("shutouts")), saves,
    )

## Loop over all players, routed by position (one small transaction each, plus a retry pass)

In [ ]:
SKATER_UPSERT = """
INSERT INTO skater_season_stats
    (player_id, season, team_id, games_played, goals, assists, points,
     plus_minus, penalty_min, shots, avg_toi)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_id=VALUES(team_id), games_played=VALUES(games_played), goals=VALUES(goals),
    assists=VALUES(assists), points=VALUES(points), plus_minus=VALUES(plus_minus),
    penalty_min=VALUES(penalty_min), shots=VALUES(shots), avg_toi=VALUES(avg_toi)
"""
GOALIE_UPSERT = """
INSERT INTO goalie_season_stats
    (player_id, season, team_id, games_played, wins, losses, ot_losses,
     save_pct, goals_against_avg, shutouts, saves)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_id=VALUES(team_id), games_played=VALUES(games_played), wins=VALUES(wins),
    losses=VALUES(losses), ot_losses=VALUES(ot_losses), save_pct=VALUES(save_pct),
    goals_against_avg=VALUES(goals_against_avg), shutouts=VALUES(shutouts), saves=VALUES(saves)
"""

counts = {"skaters": 0, "goalies": 0, "no_stats": 0}

def load_players(batch):
    failed = []
    start = time.time()
    for i, (pid, team_id, db_position) in enumerate(batch, 1):
        try:
            data = get_json(PLAYER_LANDING_URL.format(player_id=pid))
            position = data.get("position") or db_position
            if position == "G":
                row = parse_goalie_stats(data, pid, team_id)
                sql, key = GOALIE_UPSERT, "goalies"
            else:
                row = parse_skater_stats(data, pid, team_id)
                sql, key = SKATER_UPSERT, "skaters"
            if row is None:
                counts["no_stats"] += 1          # e.g. rookie with no NHL games yet
            else:
                cur.execute(sql, row)
                conn.commit()
                counts[key] += 1
        except Exception as e:
            conn.rollback()
            print(f"Player {pid}: FAILED - {type(e).__name__}: {e}")
            failed.append((pid, team_id, db_position))
        if i % 50 == 0:
            print(f"[{i}/{len(batch)}] {counts} - {time.time() - start:.0f}s elapsed")
        time.sleep(REQUEST_DELAY)
    return failed

failed_players = load_players(players_list)
if failed_players:
    print(f"\nRetrying {len(failed_players)} failed players...")
    time.sleep(3)
    failed_players = load_players(failed_players)

print(f"\nDone. {counts}")
if failed_players:
    print("Still failing (re-run this cell):", [p[0] for p in failed_players])

## Sanity checks

In [ ]:
for table in ("skater_season_stats", "goalie_season_stats"):
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"Rows in {table}:", cur.fetchone()[0])

cur.execute("""
    SELECT p.first_name, p.last_name, t.team_abbrev, s.goals, s.assists, s.points
    FROM skater_season_stats s
    JOIN players p ON s.player_id = p.player_id
    JOIN teams t ON s.team_id = t.team_id
    ORDER BY s.points DESC
    LIMIT 10
""")
display(pd.DataFrame(cur.fetchall(), columns=["first_name", "last_name", "team", "goals", "assists", "points"]))

In [ ]:
# games_played >= 5 filters out backups with a tiny sample (raise to 20+ for a real leaderboard)
cur.execute("""
    SELECT p.first_name, p.last_name, t.team_abbrev, g.wins, g.save_pct, g.goals_against_avg
    FROM goalie_season_stats g
    JOIN players p ON g.player_id = p.player_id
    JOIN teams t ON g.team_id = t.team_id
    WHERE g.games_played >= 5
    ORDER BY g.save_pct DESC
    LIMIT 10
""")
display(pd.DataFrame(cur.fetchall(), columns=["first_name", "last_name", "team", "wins", "save_pct", "gaa"]))

In [ ]:
cur.close()
conn.close()
session.close()